# SionnaRTStudio — Google Colab Launcher

This notebook launches the public [`alexandarM/SionnaRTStudio`](https://github.com/alexandarM/SionnaRTStudio) fork of the original [`puloktarafder/SionnaRTStudio`](https://github.com/puloktarafder/SionnaRTStudio). It installs required system packages, clones the selected GitHub repository and branch, runs the project's setup script, checks the frontend/backend, and opens the application through the Colab port proxy.

## Before running

1. In the configuration cell, the repository is preconfigured for `alexandarM/SionnaRTStudio` on the `main` branch. The repository must be public or otherwise accessible to the Colab runtime.
2. Select **Runtime → Change runtime type → GPU** when an NVIDIA GPU is available. Ray tracing can fall back to CPU/LLVM, but it may be much slower.
3. Run the cells from top to bottom. The first setup may take several minutes and requires network access.
4. Google Colab runtimes are temporary. The optional Google Drive integration preserves the SQLite experiment database, but measurement datasets stored in backend memory must be imported again after a backend/runtime restart.

The launcher keeps setup output in `/content/SionnaRTStudio_setup.log` and application output in `/content/SionnaRTStudio_colab.log`. It clones the configured fork and branch, not the upstream repository by default.


In [ ]:
# 1. Repository, persistence, and optional feature configuration
from pathlib import Path

GITHUB_REPOSITORY = "alexandarM/SionnaRTStudio"  # GitHub owner/repository for this fork
GITHUB_BRANCH = "main"

# Mount Google Drive to preserve the SQLite experiment database across runtime resets.
ENABLE_GOOGLE_DRIVE_PERSISTENCE = True
GOOGLE_DRIVE_DATA_DIR = Path("/content/drive/MyDrive/SionnaRTStudio/data")

# The NR PHY dependency set is large; keep this disabled unless you need PUSCH BER/BLER.
INSTALL_OPTIONAL_PHY = False

if not GITHUB_REPOSITORY:
    raise ValueError(
        "Set GITHUB_REPOSITORY to your fork, e.g. 'owner/SionnaRTStudio', before continuing."
    )
if "/" not in GITHUB_REPOSITORY or GITHUB_REPOSITORY.startswith("/") or GITHUB_REPOSITORY.endswith("/"):
    raise ValueError("GITHUB_REPOSITORY must use the 'owner/repository' format.")

REPO_URL = f"https://github.com/{GITHUB_REPOSITORY}.git"
PROJECT_DIR = Path("/content/SionnaRTStudio")
SETUP_LOG = Path("/content/SionnaRTStudio_setup.log")
APP_LOG = Path("/content/SionnaRTStudio_colab.log")

print("Repository:", REPO_URL)
print("Branch:", GITHUB_BRANCH)
print("Google Drive persistence:", ENABLE_GOOGLE_DRIVE_PERSISTENCE)
print("Optional NR PHY installation:", INSTALL_OPTIONAL_PHY)


In [ ]:
# 2. Check the Colab runtime, mount Drive if enabled, and install system packages
import os
import sys
import shutil
import subprocess

print("Python:", sys.version.split()[0])
print("Platform:", sys.platform)

has_gpu = False
if shutil.which("nvidia-smi"):
    gpu_check = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True)
    has_gpu = gpu_check.returncode == 0 and "GPU" in gpu_check.stdout
    print("nvidia-smi:")
    print((gpu_check.stdout or gpu_check.stderr).strip()[:2000])
else:
    print("nvidia-smi is not available in this runtime.")

if has_gpu:
    print("NVIDIA GPU detected. SionnaRTStudio will attempt to use the CUDA ray-tracing variant.")
else:
    print("No NVIDIA GPU was confirmed. The application may use CPU/LLVM ray tracing, which can be much slower.")

if ENABLE_GOOGLE_DRIVE_PERSISTENCE:
    from google.colab import drive
    drive.mount("/content/drive")
    GOOGLE_DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["SIONNA_RT_STUDIO_DATA_DIR"] = str(GOOGLE_DRIVE_DATA_DIR)
    print("Experiment database persistence directory:", os.environ["SIONNA_RT_STUDIO_DATA_DIR"])
else:
    os.environ.pop("SIONNA_RT_STUDIO_DATA_DIR", None)
    print("Experiment database will use the Colab runtime filesystem and will not survive a full runtime reset.")

if not Path("/content/.srts_colab_system_deps_ok").exists():
    print("Installing system packages required by setup.sh...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq",
        "git", "curl", "ca-certificates", "tar", "xz-utils",
        "python3-venv", "llvm-runtime"
    ], check=True)
    Path("/content/.srts_colab_system_deps_ok").touch()
    print("System dependencies installed.")
else:
    print("System dependencies are already installed in this runtime.")


In [ ]:
# 3. Clone or update the configured fork
import subprocess

PROJECT_DIR = Path("/content/SionnaRTStudio")

def run_git(args, *, cwd=None):
    return subprocess.run(args, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True).stdout.strip()

def normalize_git_url(url):
    return url.removesuffix("/").removesuffix(".git").lower()

if not PROJECT_DIR.exists():
    print("Cloning the configured fork...")
    run_git(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, REPO_URL, str(PROJECT_DIR)])
else:
    if not (PROJECT_DIR / ".git").exists():
        raise RuntimeError(
            f"{PROJECT_DIR} already exists but is not a Git repository. Rename or remove this directory, then rerun this cell."
        )
    current_origin = run_git(["git", "remote", "get-url", "origin"], cwd=PROJECT_DIR)
    if normalize_git_url(current_origin) != normalize_git_url(REPO_URL):
        raise RuntimeError(
            "The existing checkout points to a different Git remote. "
            f"Current origin: {current_origin}. Configured fork: {REPO_URL}. "
            "Remove /content/SionnaRTStudio or start with a fresh Colab runtime."
        )
    local_changes = run_git(["git", "status", "--porcelain"], cwd=PROJECT_DIR)
    if not local_changes:
        print("The configured fork is already present with no local changes; pulling the latest branch...")
        run_git(["git", "pull", "--ff-only", "origin", GITHUB_BRANCH], cwd=PROJECT_DIR)
    else:
        print("The checkout contains local changes. They will be preserved; automatic pull was skipped.")

commit = run_git(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT_DIR)
print("Project directory:", PROJECT_DIR)
print("Git commit:", commit)
required_files = ["setup.sh", "run.sh", "package.json", "backend/main.py", "SionnaRTStudio_Colab_Launcher.ipynb"]
missing = [name for name in required_files if not (PROJECT_DIR / name).exists()]
if missing:
    raise FileNotFoundError("The configured repository is missing required project files: " + ", ".join(missing))
print("Repository structure check passed.")


In [ ]:
# 4. Prepare a Python interpreter and run the project setup script with diagnostics
import os
import sys
import shutil
import subprocess
import tempfile
import re
from pathlib import Path

PROJECT_DIR = Path("/content/SionnaRTStudio")
SETUP_LOG = Path("/content/SionnaRTStudio_setup.log")
if not (PROJECT_DIR / "setup.sh").exists():
    raise FileNotFoundError("setup.sh was not found. Rerun the repository clone cell.")

print("=== setup.sh preflight ===")
print("Active Colab Python:", sys.version.split()[0], sys.executable)

candidate_names = ["python3.12", "python3.13", "python3.11", "python3"]
working_candidate = None
tried_versions = set()
apt_update_done = False

def venv_smoke_test(py_bin):
    temp_root = Path(tempfile.mkdtemp(prefix="srts-venv-check-", dir="/content"))
    test_venv = temp_root / "venv"
    try:
        proc = subprocess.run(
            [py_bin, "-m", "venv", str(test_venv)],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=120
        )
        ok = (proc.returncode == 0 and (test_venv / "bin" / "python").exists()
              and (test_venv / "bin" / "pip").exists())
        return ok, (proc.stdout or "")[-2500:]
    except Exception as exc:
        return False, repr(exc)
    finally:
        shutil.rmtree(temp_root, ignore_errors=True)

for candidate in candidate_names:
    candidate_path = shutil.which(candidate)
    if not candidate_path:
        continue
    try:
        version = subprocess.run(
            [candidate_path, "-c", "import sys; print(f'{sys.version_info[0]}.{sys.version_info[1]}')"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=15, check=True
        ).stdout.strip()
    except Exception:
        continue
    if version in tried_versions:
        continue
    tried_versions.add(version)
    print(f"\nChecking virtual-environment support: {candidate} ({version})...")
    ok, detail = venv_smoke_test(candidate_path)
    if not ok:
        print("The initial venv/ensurepip check failed.")
        package_name = f"python{version}-venv"
        if not apt_update_done:
            apt_update = subprocess.run(
                ["apt-get", "update", "-qq"], text=True,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300
            )
            apt_update_done = True
            if apt_update.returncode != 0:
                print("apt-get update warning:", (apt_update.stdout or "")[-1500:])
        apt_install = subprocess.run(
            ["apt-get", "install", "-y", "-qq", package_name], text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300
        )
        if apt_install.returncode == 0:
            ok, detail = venv_smoke_test(candidate_path)
        else:
            print(f"Could not install {package_name}:", (apt_install.stdout or "")[-1500:])
    if ok:
        working_candidate = candidate
        print(f"Virtual-environment check passed for {candidate} ({version}).")
        break
    print(f"{candidate} is not suitable for venv creation. Details: {detail}")

if not working_candidate:
    raise RuntimeError(
        "No available Python interpreter could create a virtual environment with pip. "
        "Review the messages above; the matching pythonX.Y-venv package may be missing."
    )

# Change only the interpreter preference in this local checkout of setup.sh.
# All other project installation commands and pinned dependencies remain unchanged.
setup_script = PROJECT_DIR / "setup.sh"
setup_text = setup_script.read_text()
match = re.search(r"for cand in ([^;]+); do", setup_text)
if not match:
    raise RuntimeError("Could not identify the Python candidate list in setup.sh; refusing to patch it automatically.")
original_candidates = [item for item in match.group(1).split() if item in candidate_names]
ordered_candidates = [working_candidate] + [item for item in original_candidates if item != working_candidate]
ordered_candidates += [item for item in candidate_names if item not in ordered_candidates]
patched_line = "for cand in " + " ".join(ordered_candidates) + "; do"
if match.group(0) != patched_line:
    setup_text = setup_text[:match.start()] + patched_line + setup_text[match.end():]
    setup_script.write_text(setup_text)
    print("Updated only the Python interpreter preference in the local setup.sh checkout:", ordered_candidates)
else:
    print("setup.sh already uses the selected interpreter preference:", ordered_candidates)

for tool in ("python3.13", "python3.12", "python3.11", "node", "npm", "curl", "git"):
    tool_path = shutil.which(tool)
    if tool_path:
        version_result = subprocess.run([tool_path, "--version"], text=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, timeout=15)
        print(f"{tool}: {tool_path} | {(version_result.stdout or '').strip()[:180]}")
    else:
        print(f"{tool}: not found")

install_env = os.environ.copy()
install_env.setdefault("SRTS_NODE_VERSION", "22.23.1")
SETUP_LOG.write_text("")
command = f"bash setup.sh 2>&1 | tee {str(SETUP_LOG)}"
print("\nRunning setup.sh. Full output is being saved to:", SETUP_LOG)
print("If installation fails, the last 160 log lines will be displayed.\n")
result = subprocess.run(
    ["bash", "-o", "pipefail", "-c", command],
    cwd=PROJECT_DIR, env=install_env, text=True,
)

if result.returncode != 0:
    log_text = SETUP_LOG.read_text(errors="replace") if SETUP_LOG.exists() else ""
    tail = "\n".join(log_text.splitlines()[-160:])
    lower = log_text.lower()
    if "npm error" in lower or "npm err!" in lower:
        stage = "Frontend npm installation. Review the npm error in the log."
    elif ("no matching distribution found" in lower or "could not find a version that satisfies" in lower
          or "error: subprocess-exited-with-error" in lower or "failed building wheel" in lower):
        stage = "Python backend dependencies or wheel installation. Check the Python version and pip output."
    elif "curl:" in lower or "checksum mismatch" in lower or ("downloading node" in lower and "error" in lower):
        stage = "Downloading the local Node.js runtime or verifying its checksum."
    elif "could not create a venv" in lower or "no module named venv" in lower:
        stage = "Creating the Python virtual environment."
    elif "neither gpu nor cpu ray tracing" in lower or "mitsuba could not be imported" in lower:
        stage = "Initializing the Mitsuba/Dr.Jit GPU or LLVM backend."
    else:
        stage = "setup.sh stopped; review the final log lines below."
    print("\nInstallation failed.")
    print("Likely stage:", stage)
    print("Log file:", SETUP_LOG)
    print("\n========== END OF SETUP LOG (last 160 lines) ==========")
    print(tail if tail else "The log is empty; setup.sh produced no output.")
    raise RuntimeError(
        f"setup.sh exited with code {result.returncode}. The exit code alone does not identify the root cause. "
        f"Full output is saved in {SETUP_LOG}. Review the first actual ERROR message in the log."
    )

PYTHON_BIN = PROJECT_DIR / ".venv" / "bin" / "python"
NODE_BIN = PROJECT_DIR / ".node" / "bin" / "node"
if not PYTHON_BIN.exists():
    raise FileNotFoundError(f"Project Python interpreter is missing: {PYTHON_BIN}")
if not (PROJECT_DIR / "node_modules" / ".bin" / "vite").exists():
    raise FileNotFoundError("Frontend dependencies are missing (node_modules/.bin/vite). Review the setup log.")

print("\nsetup.sh completed successfully.")
print("Backend Python:", PYTHON_BIN)
if NODE_BIN.exists():
    print("Project-local Node.js:", NODE_BIN)


## Optional: enable 5G NR PUSCH BER/BLER experiments

The core application works without the separate NR PHY package. Its dependencies are large and may require several gigabytes of additional downloads. Set `INSTALL_OPTIONAL_PHY = True` in the first configuration cell and run the next cell **before starting the application** only when you need the **Run BER Sweep** workflow. Leave it disabled for the GUI, radio maps, ray tracing, mobility, and export features.

In [ ]:
# 5. Install the optional NR PHY dependency set when requested
import subprocess

if INSTALL_OPTIONAL_PHY:
    requirements_file = PROJECT_DIR / "backend" / "requirements-phy.txt"
    if not requirements_file.exists():
        raise FileNotFoundError(f"Optional PHY requirements file was not found: {requirements_file}")
    print("Installing the optional Sionna PHY package. This requires additional download time and disk space...")
    subprocess.run([
        str(PROJECT_DIR / ".venv" / "bin" / "python"), "-m", "pip", "install", "-r", str(requirements_file)
    ], check=True)
    print("Optional PHY package installed. Start or restart the server after this cell.")
else:
    print("Skipping optional PHY dependencies. Core features remain available; BER/BLER may be unavailable.")


In [ ]:
# 6. Run installation checks, backend tests, TypeScript checks, and a production build
import os
import subprocess

PROJECT_DIR = Path("/content/SionnaRTStudio")
PYTHON_BIN = str(PROJECT_DIR / ".venv" / "bin" / "python")
test_env = os.environ.copy()
if (PROJECT_DIR / ".node" / "bin").exists():
    test_env["PATH"] = str(PROJECT_DIR / ".node" / "bin") + os.pathsep + test_env.get("PATH", "")

# pytest is a test-only dependency and is intentionally not part of the runtime install.
pytest_probe = subprocess.run(
    [PYTHON_BIN, "-c", "import pytest"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
if pytest_probe.returncode != 0:
    print("Installing pytest in the project virtual environment for the full backend test suite...")
    subprocess.run([PYTHON_BIN, "-m", "pip", "install", "pytest"], check=True)

commands = [
    ("Backend pytest suite", [PYTHON_BIN, "-m", "pytest", "-q", "backend/tests"]),
    ("OSM parser check", ["npm", "run", "check:osm-parser"]),
    ("Frontend TypeScript check", ["npm", "run", "lint"]),
    ("Frontend production build", ["npm", "run", "build"]),
]

failed_checks = []
for label, command in commands:
    print(f"\n=== {label} ===")
    try:
        result = subprocess.run(
            command, cwd=PROJECT_DIR, env=test_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600
        )
        print(result.stdout[-14000:] if result.stdout else "(no output)")
        if result.returncode == 0:
            print(f"PASS: {label}")
        else:
            print(f"FAIL: {label} (exit={result.returncode})")
            failed_checks.append(label)
    except subprocess.TimeoutExpired:
        print(f"TIMEOUT: {label} exceeded 600 seconds.")
        failed_checks.append(label)

if failed_checks:
    print("Checks requiring review:", ", ".join(failed_checks))
else:
    print("All configured checks passed.")


In [ ]:
# 7. Start the backend and frontend, then verify /api/health
import os
import sys
import time
import json
import socket
import signal
import subprocess
import urllib.request
from pathlib import Path
from IPython.display import display, HTML

PROJECT_DIR = Path("/content/SionnaRTStudio")
APP_LOG = Path("/content/SionnaRTStudio_colab.log")
FRONTEND_PORT = 3000

if not (PROJECT_DIR / ".venv" / "bin" / "python").exists():
    raise RuntimeError("The project virtual environment is missing. Complete the setup cell first.")

def get_json(url, timeout=3):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            raw = response.read().decode("utf-8", errors="replace")
            return json.loads(raw), response.status
    except Exception:
        return None, None

def get_status(url, timeout=3):
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "SionnaRTStudio-Colab-Launcher"})
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.status
    except Exception:
        return None

def port_is_free(port, host="127.0.0.1"):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex((host, port)) != 0

known_port = globals().get("BACKEND_PORT")
known_health = None
if known_port:
    known_health, _ = get_json(f"http://127.0.0.1:{int(known_port)}/api/health")
existing_health, _ = get_json("http://127.0.0.1:8000/api/health")
existing_frontend = get_status(f"http://127.0.0.1:{FRONTEND_PORT}/")
existing_proxy_health, _ = get_json(f"http://127.0.0.1:{FRONTEND_PORT}/api/health")

if (known_health and known_health.get("app") == "SionnaRTStudio" and existing_frontend == 200
        and existing_proxy_health and existing_proxy_health.get("app") == "SionnaRTStudio"):
    BACKEND_PORT = int(known_port)
    print("SionnaRTStudio is already running in this kernel. Reusing the existing server.")
elif (existing_health and existing_health.get("app") == "SionnaRTStudio" and existing_frontend == 200
      and existing_proxy_health and existing_proxy_health.get("app") == "SionnaRTStudio"):
    BACKEND_PORT = 8000
    print("SionnaRTStudio is already running. Reusing the existing server.")
else:
    if not port_is_free(FRONTEND_PORT):
        raise RuntimeError(
            f"Port {FRONTEND_PORT} is occupied, but it does not appear to serve SionnaRTStudio. "
            "A fresh Colab runtime usually has this port free. Restart the runtime and rerun the notebook."
        )
    available_backend_ports = [port for port in range(8000, 8020) if port_is_free(port)]
    if not available_backend_ports:
        raise RuntimeError("No free backend port was found in the range 8000–8019. Restart the Colab runtime.")
    BACKEND_PORT = 8000 if 8000 in available_backend_ports else available_backend_ports[0]

    run_env = os.environ.copy()
    run_env["SRTS_BACKEND_PORT"] = str(BACKEND_PORT)
    if ENABLE_GOOGLE_DRIVE_PERSISTENCE:
        run_env["SIONNA_RT_STUDIO_DATA_DIR"] = str(GOOGLE_DRIVE_DATA_DIR)
    if (PROJECT_DIR / ".node" / "bin").exists():
        run_env["PATH"] = str(PROJECT_DIR / ".node" / "bin") + os.pathsep + run_env.get("PATH", "")

    APP_LOG.parent.mkdir(parents=True, exist_ok=True)
    SRTS_LOG_HANDLE = open(APP_LOG, "ab", buffering=0)
    SRTS_PROCESS = subprocess.Popen(
        ["bash", "run.sh"], cwd=PROJECT_DIR, env=run_env,
        stdout=SRTS_LOG_HANDLE, stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, start_new_session=True
    )
    print(f"Started run.sh (PID={SRTS_PROCESS.pid}). Log: {APP_LOG}")
    print("Waiting for the backend and frontend health checks...")

    deadline = time.time() + 300
    health = None
    frontend_status = None
    while time.time() < deadline:
        health, _ = get_json(f"http://127.0.0.1:{BACKEND_PORT}/api/health", timeout=2)
        frontend_status = get_status(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=2)
        proxied_health, _ = get_json(f"http://127.0.0.1:{FRONTEND_PORT}/api/health", timeout=2)
        if (health and health.get("app") == "SionnaRTStudio" and frontend_status == 200
                and proxied_health and proxied_health.get("app") == "SionnaRTStudio"):
            break
        if SRTS_PROCESS.poll() is not None:
            break
        time.sleep(2)

    health, _ = get_json(f"http://127.0.0.1:{BACKEND_PORT}/api/health", timeout=3)
    frontend_status = get_status(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=3)
    proxied_health, _ = get_json(f"http://127.0.0.1:{FRONTEND_PORT}/api/health", timeout=3)
    if not (health and health.get("app") == "SionnaRTStudio" and frontend_status == 200
            and proxied_health and proxied_health.get("app") == "SionnaRTStudio"):
        print("\nThe application did not pass its health checks. Recent application log:")
        try:
            print(APP_LOG.read_text(errors="replace")[-18000:])
        except Exception as exc:
            print("Could not read the application log:", exc)
        raise RuntimeError(
            "SionnaRTStudio did not become ready within five minutes. The log above usually identifies the cause "
            "(for example CUDA/LLVM, npm installation, or network access)."
        )

health, _ = get_json(f"http://127.0.0.1:{BACKEND_PORT}/api/health")
print("\nBackend and frontend are online. /api/health:")
print(json.dumps(health, indent=2, ensure_ascii=False))

try:
    from google.colab import output
    app_url = output.eval_js(f"google.colab.kernel.proxyPort({FRONTEND_PORT})")
    print("\nOpen SionnaRTStudio:", app_url)
    display(HTML(
        '<div style="padding:14px;border:1px solid #bbb;border-radius:10px;">'
        f'<a href="{app_url}" target="_blank" rel="noopener noreferrer" '
        'style="font-size:18px;font-weight:600;">Open SionnaRTStudio in a new tab</a>'
        '<p style="margin-bottom:0">Confirm that the application header reports RT-CORE ONLINE.</p>'
        '</div>'
    ))
except Exception as exc:
    print("The Colab proxy URL could not be displayed automatically:", exc)
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(FRONTEND_PORT)
    except Exception:
        print("Open the frontend using the Colab port proxy for port 3000.")


In [ ]:
# 8. Display recent application logs when troubleshooting
from pathlib import Path

APP_LOG = Path("/content/SionnaRTStudio_colab.log")
if APP_LOG.exists():
    log_text = APP_LOG.read_text(errors="replace")
    print(log_text[-25000:] if log_text else "The application log is empty.")
else:
    print("The application log does not exist yet. Run the start cell first.")


In [ ]:
# 9. Stop the SionnaRTStudio process group when finished
import os
import signal
import time

process = globals().get("SRTS_PROCESS")
if process is None:
    print("No SRTS_PROCESS handle is available in this kernel. If the runtime was reset, its processes have already stopped.")
elif process.poll() is not None:
    print("The SionnaRTStudio process has already exited.")
else:
    try:
        os.killpg(process.pid, signal.SIGTERM)
        time.sleep(2)
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGKILL)
        print("A stop signal was sent to the SionnaRTStudio process group.")
    except ProcessLookupError:
        print("The application processes have already stopped.")


## Troubleshooting and operational notes

- **RT-CORE OFFLINE:** inspect `/content/SionnaRTStudio_colab.log` and the `/api/health` response. Mitsuba must initialize a usable CUDA or LLVM variant.
- **CPU instead of GPU:** select **Runtime → Change runtime type → GPU**, restart the runtime, and rerun the notebook. If no GPU is available, CPU/LLVM fallback may work but will be slower.
- **Setup fails:** review `/content/SionnaRTStudio_setup.log`. Colab network access or package mirrors may be temporarily unavailable. Rerun the failed cell after addressing the first actual error.
- **BER/BLER controls are unavailable:** the optional Sionna PHY dependencies were not installed. Enable `INSTALL_OPTIONAL_PHY`, run its installation cell, and restart the application.
- **Experiment history disappears after a runtime reset:** confirm that Google Drive persistence was enabled and that the application was started with `SIONNA_RT_STUDIO_DATA_DIR` pointing to the Drive data directory.
- **Measurement data disappears after a backend restart:** this is expected for the current in-memory measurement store. Re-import the measurement CSV files.
- **The Colab session expires:** reconnect the runtime and run the notebook again. Colab is temporary compute, not persistent application hosting.
- **Privacy:** do not use this unauthenticated application for sensitive data. The Colab port proxy is intended for your active notebook session; the application itself does not add user authentication.

The launcher prints the selected repository, branch, commit, and `/api/health` response for reproducibility. It only patches the Python interpreter preference in the local Colab checkout when needed for virtual-environment creation; it does not rewrite the upstream or fork repository files on GitHub.
